# Volume 5, Chapter 7: The explicit formula and the Riemann hypothesis (Python companion)

Exercises first, solutions at the end. We rebuild $\psi(x)$ from the zeros of $\zeta$ (Theorem 7.5), measure the error term of the prime number theorem up to $10^7$ against what the Riemann hypothesis predicts (Theorem 7.10), compare $\pi(x)$ with $\operatorname{Li}(x)$ and with Riemann's function, run the prime races modulo $3$, $4$ and $8$ (Proposition 7.11, Exercise 7.6), test the smoothed formula of Exercise 7.7, and end with signal processing: the Fourier transform of the primes shows the zeros. Requirements: numpy, mpmath.

*Before each solution, predict what the output will look like.*

## Exercises

**P7.1 (★) Music.** Compute the first $200$ zeros and evaluate the right side of the explicit formula (Corollary 7.6) with $10$, $50$ and $200$ pairs of zeros at a few points, including a prime power. Compare with $\psi_0(x)$.

**P7.2 (★) How small is the error?** Sieve up to $10^7$ and compute $\max\frac{|\psi(x)-x|}{\sqrt x}$ on each decade. Compare $\frac{|\psi(x)-x|}{\sqrt x\log^2x}$ with Schoenfeld's constant $\frac1{8\pi}$.

**P7.3 (★) Gauss against Riemann.** Tabulate $\operatorname{Li}(x)-\pi(x)$ and $\operatorname{Ri}(x)-\pi(x)$, where $\operatorname{Ri}(x)=\sum_n\frac{\mu(n)}n\operatorname{li}(x^{1/n})$ (`mpmath.riemannr`), for $x=10^3,\dots,10^7$, next to $\sqrt x\log x$ (Exercise 7.5).

**P7.4 (★★) Races.** Find the first $x$ with $\pi(x;4,1)>\pi(x;4,3)$. Average $\frac{\theta(x;4,3)-\theta(x;4,1)}{\sqrt x}$ over $\log x\in[\log10^3,\log10^7]$, do the same modulo $3$, and test the modulo $8$ prediction of Exercise 7.6.

**P7.5 (★) The smoothed formula.** Check Exercise 7.7 at $x=10,100,1000,10^4$ with $10$ and $200$ pairs of zeros. Why does it converge so much faster than P7.1?

**P7.6 (★★) CS application: hearing the zeros.** Sample $f(u)=(\psi(e^u)-e^u)e^{-u/2}$ on $\log10\le u\le\log10^7$, apply a window and a fast Fourier transform, and list the strongest frequencies between $5$ and $60$.

## Solutions

### P7.1

In [ ]:
import math, cmath
import numpy as np
import mpmath as mp
mp.mp.dps = 15

K = 200
gammas = [float(mp.zetazero(n).imag) for n in range(1, K + 1)]
print(f"{K} zeros, the last at height {gammas[-1]:.2f}")

N = 10**7
is_p = np.ones(N + 1, dtype=bool); is_p[:2] = False
for p in range(2, int(N**0.5) + 1):
    if is_p[p]: is_p[p*p::p] = False
primes = np.nonzero(is_p)[0]
Lam = np.zeros(N + 1)
for p in primes[primes <= int(N**0.5)]:
    q = p * p
    while q <= N: Lam[q] = math.log(p); q *= p
Lam[primes] = np.log(primes)
psi = np.cumsum(Lam)

def psi0(x):
    n = int(math.floor(x))
    return psi[n] - (Lam[n] / 2 if n == x else 0)

def explicit(x, k):
    s = x - math.log(2 * math.pi) - 0.5 * math.log(1 - x**-2)
    for g in gammas[:k]:
        rho = complex(0.5, g)
        s -= 2 * (cmath.exp(rho * math.log(x)) / rho).real
    return s

print("     x    psi0(x)   10 pairs   50 pairs  200 pairs")
for x in (10.5, 19, 30.5, 100.5, 101):
    print(f"{x:6}  {psi0(x):9.4f}  {explicit(x, 10):9.4f}  {explicit(x, 50):9.4f}  {explicit(x, 200):9.4f}")

The approximations improve slowly and not monotonically: the series converges only conditionally, like a Fourier series with a jump. At the prime $19$ and the prime $101$, $\psi_0$ is the midpoint of the jump, and the truncated sums aim at it. Each extra zero adds a wave of amplitude $\frac{2\sqrt x}{|\rho|}$, so at $x=100$ even the $200$th zero still moves the sum by a few hundredths.

### P7.2

In [ ]:
xs = np.arange(2, N + 1, dtype=float)
# just before each jump psi takes the value psi[n-1], so test both sides of every integer
err = np.maximum(np.abs(psi[2:] - xs), np.abs(psi[1:-1] - xs)) / np.sqrt(xs)
for a, b in ((10**3, 10**4), (10**4, 10**5), (10**5, 10**6), (10**6, 10**7)):
    seg = err[a - 2:b - 2]
    print(f"max |psi(x)-x|/sqrt(x) on [{a:.0e}, {b:.0e}]: {seg.max():.3f}")
ratio = err / np.log(xs)**2
print(f"max |psi(x)-x|/(sqrt(x) log^2 x) for 73 <= x <= 1e7: {ratio[71:].max():.4f};  Schoenfeld's 1/(8 pi) = {1/(8*math.pi):.4f}")

The error stays below $0.81\sqrt x$ on the whole range, and the ratio to $\sqrt x\log^2x$ stays under Schoenfeld's conditional bound. This is what Theorem 7.10 predicts if the Riemann hypothesis holds; the factor $\log^2x$ there comes from summing $\frac1{|\rho|}$ over the zeros, the worst case in which all the waves agree, and in practice they mostly cancel. Littlewood's theorem says the ratio to $\sqrt x$ alone is unbounded, but it grows like $\log\log\log x$: invisible at any height a computer can reach.

### P7.3

In [ ]:
pi_count = np.cumsum(is_p)
print("      x       pi(x)    Li(x)-pi(x)   Ri(x)-pi(x)   sqrt(x)log(x)")
for k in range(3, 8):
    x = 10**k
    Li = float(mp.li(x) - mp.li(2))
    Ri = float(mp.riemannr(x))
    print(f"{x:9d} {pi_count[x]:9d} {Li - pi_count[x]:12.1f} {Ri - pi_count[x]:12.1f} {math.sqrt(x)*math.log(x):14.0f}")

$\operatorname{Li}(x)$ overshoots by an amount that grows roughly like $\frac{\sqrt x}{\log x}$: that is the term $\frac12\operatorname{li}(\sqrt x)$ coming from the squares of primes, the same effect as in Chebyshev's bias. Riemann's $\operatorname{Ri}$ removes it (the $n=2$ term is $-\frac12\operatorname{li}(\sqrt x)$) and is far more accurate. What remains is the contribution of the zeros, $-\sum_\rho\operatorname{li}(x^\rho)$ and its relatives, of size $\frac{\sqrt x}{\log x}$ with oscillating sign. Everything is far below the bound $\sqrt x\log x$ of Exercise 7.5.

### P7.4

In [ ]:
def theta_class(q, a):
    w = np.zeros(N + 1); sel = primes[primes % q == a]; w[sel] = np.log(sel)
    return np.cumsum(w)
c1 = np.cumsum(is_p & (np.arange(N + 1) % 4 == 1))
c3 = np.cumsum(is_p & (np.arange(N + 1) % 4 == 3))
lead1 = np.nonzero(c1[3:] > c3[3:])[0] + 3
print(f"pi(x;4,1) > pi(x;4,3) first at x = {lead1[0]}, and at {len(lead1)} integers x <= 1e7 in all")
t41, t43 = theta_class(4, 1), theta_class(4, 3)
us = np.linspace(math.log(1e3), math.log(1e7), 40001)
xu = np.exp(us).astype(int)
E = (t43[xu] - t41[xu]) / np.sqrt(xu)
print(f"mod 4: average of (theta(x;4,3)-theta(x;4,1))/sqrt(x) over log x in [log 1e3, log 1e7]: {E.mean():.3f}  (prediction 1)")
t31, t32 = theta_class(3, 1), theta_class(3, 2)
E3 = (t32[xu] - t31[xu]) / np.sqrt(xu)
print(f"mod 3: average of (theta(x;3,2)-theta(x;3,1))/sqrt(x): {E3.mean():.3f}  (prediction 1)")
theta = np.cumsum(np.where(is_p, np.log(np.maximum(np.arange(N + 1), 1)), 0.0))
for a in (1, 3, 5, 7):
    t = theta_class(8, a)
    Ea = (t[xu] - theta[xu] / 4) / np.sqrt(xu)
    print(f"mod 8, a = {a}: average of (theta(x;8,a) - theta(x)/4)/sqrt(x) = {Ea.mean():+.3f}   (prediction {(-0.75 if a == 1 else 0.25):+.2f})")

Modulo $4$, class $1$ leads at only $3406$ integers up to $10^7$, the first time at Leech's $26861$. The averages over $\log x$ come close to the predicted bias $1$ (the oscillation has mean $0$ only in the limit, and four decades of $\log x$ are a short window for waves of period $\frac{2\pi}{6.02}\approx1.04$). Modulo $8$, the prediction $-\frac34,\frac14,\frac14,\frac14$ of Exercise 7.6 is matched to two decimals: class $1$, which receives all the odd squares in the prime power count, trails in the prime count.

### P7.5

In [ ]:
zp = float(mp.zeta(-1, derivative=1) / mp.zeta(-1))
def cesaro_lhs(x):
    n = np.arange(2, int(x) + 1)
    return float(np.sum(Lam[n] * (1 - n / x)))
def cesaro_rhs(x, k):
    s = x / 2 - math.log(2 * math.pi) + zp / x - sum(x**(-2*j) / (2*j*(2*j - 1)) for j in range(1, 30))
    for g in gammas[:k]:
        rho = complex(0.5, g)
        s -= 2 * (cmath.exp(rho * math.log(x)) / (rho * (rho + 1))).real
    return s
print(f"zeta'(-1)/zeta(-1) = {zp:.5f}")
for x in (10, 100, 1000, 10**4):
    print(f"x = {x:5d}: left {cesaro_lhs(x):.5f}, right with 10 pairs {cesaro_rhs(x, 10):.5f}, with 200 pairs {cesaro_rhs(x, 200):.5f}")

Each zero now contributes $\frac{x^\rho}{\rho(\rho+1)}$, of size $\frac{\sqrt x}{\gamma^2}$ instead of $\frac{\sqrt x}\gamma$, so the series converges absolutely and ten pairs already give three or four correct digits. Smoothing the cut-off in $n$ is the same as damping the high frequencies $\gamma$: a general principle of analysis (and of signal processing) that will come back in Chapter 8 and in Part III.

### P7.6

In [ ]:
u = np.arange(math.log(10.0), math.log(float(N)), 0.002)
x = np.exp(u)
f = (psi[x.astype(int)] - x) / np.sqrt(x)
f = (f - f.mean()) * np.hanning(len(f))
M = 1 << 20
spec = np.abs(np.fft.rfft(f, M))
freq = 2 * math.pi * np.fft.rfftfreq(M, d=0.002)
band = (freq > 5) & (freq < 60)
fb, sb = freq[band], spec[band]
peaks = [i for i in range(1, len(sb) - 1) if sb[i] > sb[i-1] and sb[i] > sb[i+1]]
peaks = sorted(peaks, key=lambda i: -sb[i])[:8]
print("the 8 strongest frequencies in (psi(e^u) - e^u) e^(-u/2):", sorted(round(float(fb[i]), 2) for i in peaks))
print("ordinates of the first 8 zeros:                         ", [round(g, 2) for g in gammas[:8]])

The eight strongest frequencies are, to two decimals, the ordinates of the first eight zeros. In the variable $u=\log x$, the explicit formula says $f(u)\approx-\sum_\rho\frac{e^{i\gamma u}}\rho$, a sum of pure tones at the frequencies $\gamma$, with amplitudes $\frac2{|\rho|}$ that decrease slowly. The window (Hann) trades a little resolution for much smaller side lobes, and zero-padding to $2^{20}$ points samples the spectrum finely. So the zeros of $\zeta$ can be computed from the primes, not only the primes from the zeros. With primes up to $10^7$ the frequency resolution is about $\frac{2\pi}{\log(10^6)}\approx0.45$; to separate close zeros higher up one needs exponentially more primes.